# Benchmarking Lapa: 

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- 1. Configuration ---
MODEL_ID = "lapa-llm/lapa-v0.1.2-instruct"
# MODEL_ID = "google/gemma-3-4b-it"
# MODEL_ID = "INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0"
# MODEL_ID = "google/gemma-3-12b-it"
torch_dtype = torch.bfloat16 

print(f"Loading {MODEL_ID} in {torch_dtype}...")

# --- 2. Load Model & Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch_dtype,
    attn_implementation="flash_attention_2"  # significantly speeds up inference on DGX
)
print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading lapa-llm/lapa-v0.1.2-instruct in torch.bfloat16...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1065/1065 [02:05<00:00,  8.47it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weig


Model loaded successfully!


In [2]:
LABELS = ["A", "B", "C", "D", "E", "F"]
# 1. Build the bias dictionary
# Keys must be TUPLES: (token_id,) -> float_bias
sequence_bias = {}

for l in LABELS:
    # Encode "A" and " A" (with space) without special tokens (BOS)
    token_raw = tokenizer.encode(l, add_special_tokens=False)[0]
    token_space = tokenizer.encode(" " + l, add_special_tokens=False)[0]
    
    # Apply bias (Tuple key is required!)
    sequence_bias[(token_raw,)] = 50.0
    sequence_bias[(token_space,)] = 50.0

# --- 3. Inference Function ---
def predict_answer(context, question, options, model=model, tokenizer=tokenizer):
    """
    Formats the input using the Lapa/Gemma chat template and generates an answer.
    """
    # 1. Format the User Prompt (Aligns with the fine-tuning format I provided earlier)
    options_text = "\n".join([f"{LABELS[i]}. {opt}" for i, opt in enumerate(options)])
    
    user_prompt = (
        f"Прочитай наступний текст та дай відповідь на питання.\n\n"
        f"Контекст:\n{context}\n\n"
        f"Питання:\n{question}\n\n"
        f"Варіанти відповідей:\n{options_text}\n\n"
        f"Напиши літеру правильної відповіді (літера {' '.join(LABELS)})."
    )
    
    messages = [
        {"role": "user", "content": user_prompt}
    ]
        
    # 2. Apply Chat Template
    # add_generation_prompt=True adds the "<start_of_turn>model" token
    inputs = tokenizer.apply_chat_template(
        messages, 
        return_tensors="pt", 
        add_generation_prompt=True,
        return_dict=True,
    ).to(model.device)

    # 3. Generate
    # We only need 10 new tokens max (the answer is usually just a number "1")
    outputs = model.generate(
        **inputs, 
        max_new_tokens=1, 
        do_sample=False,   # Greedy decoding for deterministic testing
        temperature=None,  
        top_p=None, sequence_bias=sequence_bias
    )

    # 4. Decode output (skipping the input prompt)
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    return generated_text.strip()

# --- 4. Run a Test Case ---
# Example: Using a dummy Ukrainian context
test_context = "Київ — столиця та найбільше місто України. Розташований на річці Дніпро. Київ є важливим промисловим, науковим, освітнім і культурним центром Східної Європи."
test_question = "На якій річці розташований Київ?"
test_options = ["Дунай", "Дніпро", "Дністер", "Десна"]

print("-" * 30)
print("Input Question:")
print(f"Q: {test_question}")
print(f"Options: {test_options}")
print("-" * 30)

answer = predict_answer(test_context, test_question, test_options)

print(f"Model Answer: {answer}")
print("-" * 30)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


------------------------------
Input Question:
Q: На якій річці розташований Київ?
Options: ['Дунай', 'Дніпро', 'Дністер', 'Десна']
------------------------------
Model Answer: B
------------------------------


In [3]:
import joblib
input_df = joblib.load("../data/dev_questions_with_search_results.joblib")
input_df.head(1)

,Question_ID,Domain,N_Pages,Question,A,B,C,D,E,F,...,Page_Num,retrieved_docs,reranked_retrieved_docs,recall_at_1,recall_at_3,recall_at_5,recall_at_10,correct_doc,page_score,retrieved_docs_simplified
0,0,domain_2,5,Як рекомендовано приймати ретаболіл дорослим?,внутрішньо,підшкірно,орально,внутрішньовенно,внутрішньом’язово,інгаляційно,...,1,"[{'page_number': 1, 'doc_id': '4e779acee13fa6e...","[{'page_number': 1, 'doc_id': '4e779acee13fa6e...",1,1,1,1,1,1.0,"[{'page_number': 1, 'doc_id': '4e779acee13fa6e..."


In [5]:
from tqdm import tqdm
predictions = []
n_context = []

N_CONTEXT = 3
MIN_SCORE = 0.5

for _, row in tqdm(input_df.iterrows(), total=len(input_df)):
    # test_context = row["reranked_retrieved_docs"][0]["full_text"]
    test_context = ""
    for i, res in enumerate(row["reranked_retrieved_docs"][:N_CONTEXT]):
        if res["reranking_score"] > MIN_SCORE:
            test_context += res["full_text"] + "\n"
        else:
            n_context.append(i)
            break
        n_context.append(i)

    test_question = row["Question"]
    test_options = [row[l] for l in LABELS]
    answer = predict_answer(test_context, test_question, test_options)
    predictions.append(answer)

100%|████████████████████████████████████████████████████████████████████████████████████████████| 461/461 [05:45<00:00,  1.33it/s]


In [6]:
input_df["pred_answer"] = [a[0] for a in predictions]
input_df["answer_is_correct"] = input_df["pred_answer"] == input_df["Correct_Answer"]

In [7]:
input_df["answer_is_correct"].mean()

np.float64(0.9240780911062907)

In [9]:
import pandas as pd
pd.Series(n_context).describe()

count    1344.000000
mean        0.973214
std         0.811788
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max         2.000000
dtype: float64

In [20]:
joblib.dump(input_df, "../data/dev_questions_with_search_results_and_prediction.joblib")

['../data/dev_questions_with_search_results_and_prediction.joblib']

# Check GUFF:

In [1]:
import joblib
input_df = joblib.load("../data/dev_questions_with_search_results.joblib")
input_df.head(1)

,Question_ID,Domain,N_Pages,Question,A,B,C,D,E,F,...,Page_Num,retrieved_docs,reranked_retrieved_docs,recall_at_1,recall_at_3,recall_at_5,recall_at_10,correct_doc,page_score,retrieved_docs_simplified
0,0,domain_2,5,Як рекомендовано приймати ретаболіл дорослим?,внутрішньо,підшкірно,орально,внутрішньовенно,внутрішньом’язово,інгаляційно,...,1,"[{'page_number': 1, 'doc_id': '4e779acee13fa6e...","[{'page_number': 1, 'doc_id': '4e779acee13fa6e...",1,1,1,1,1,1.0,"[{'page_number': 1, 'doc_id': '4e779acee13fa6e..."


In [13]:
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir

# python -m llama_cpp.server --model /root/.cache/huggingface/hub/models--lapa-llm--lapa-v0.1.2-instruct-GGUF/snapshots/19e062070c68c4a06020d09b65481567b7152947/lapa-v0.1.2-instruct-Q4_K_M.gguf --n_gpu_layers -1 --n_ctx 16384 --chat_format gemma

In [3]:
import llama_cpp
from llama_cpp import llama_cpp as lc

print("llama-cpp-python version:", getattr(llama_cpp, "__version__", "unknown"))
print("supports_gpu_offload:", lc.llama_supports_gpu_offload())

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GB10, compute capability 12.1, VMM: yes


llama-cpp-python version: 0.3.16
supports_gpu_offload: True


In [4]:
from huggingface_hub import hf_hub_download

repo_id = "lapa-llm/lapa-v0.1.2-instruct-GGUF"
filename = "lapa-v0.1.2-instruct-Q4_K_M.gguf"  # try Q4_K_M first

model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(model_path)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/root/.cache/huggingface/hub/models--lapa-llm--lapa-v0.1.2-instruct-GGUF/snapshots/19e062070c68c4a06020d09b65481567b7152947/lapa-v0.1.2-instruct-Q4_K_M.gguf


In [15]:
import openai # pip install openai
from tqdm import tqdm

# Connect to your local llama-cpp server
client = openai.OpenAI(
    base_url="http://localhost:8000/v1", 
    api_key="sk-no-key-required"
)

predictions = []
LABELS = ["A", "B", "C", "D", "E", "F"]

for _, row in tqdm(input_df.iterrows(), total=len(input_df)):
    test_context = row["reranked_retrieved_docs"][0]["full_text"]
    test_question = row["Question"]
    test_options = [row[l] for l in LABELS]
    
    options_text = "\n".join([f"{LABELS[i]}. {opt}" for i, opt in enumerate(test_options)])
    
    user_prompt = (
        f"Прочитай наступний текст та дай відповідь на питання.\n\n"
        f"Контекст:\n{test_context}\n\n"
        f"Питання:\n{test_question}\n\n"
        f"Варіанти відповідей:\n{options_text}\n\n"
        f"Напиши тільки літеру правильної відповіді (літера {' '.join(LABELS)})." 
    )
    
    try:
        response = client.chat.completions.create(
            model="gemma-3", # Name doesn't matter for local server
            messages=[{"role": "user", "content": user_prompt}],
            max_tokens=5,
            temperature=0.0
        )
        answer = response.choices[0].message.content.strip()
        predictions.append(answer)
    except Exception as e:
        print(f"Error: {e}")
        predictions.append("")

100%|████████████████████████████████████████████████████████████████████████████████████████████| 461/461 [03:42<00:00,  2.07it/s]


In [16]:
input_df["pred_answer"] = [a[0] for a in predictions]
input_df["answer_is_correct"] = input_df["pred_answer"] == input_df["Correct_Answer"]
input_df["answer_is_correct"].mean()

np.float64(0.9088937093275488)

Metrics: 
- Lapa 12b full + 3 results: 0.9240
- Lapa 12b full + 1 result: 0.9154
- Lapa GUFF + 1 result: 0.9089
- google/gemma-3-12b-it + 1 result: 0.9067
- google/gemma-3-4b-it + 1 result: 0.85
- INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0 + 1 result: 0.8373  -> LoL (worse than base model :) )